### Import Libraries

In [ ]:
import pandas as pd

from datasets import load_dataset

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## Exercise

Get accustomed to the Revolutionary Crossroads data and `Dataset`s.

### RevRoad `Dataset` Locations:

- https://huggingface.co/RevolutionCrossroads
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810 (247 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810_issues (707 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files (9.4 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs (4.23 TB)
- https://huggingface.co/datasets/RevolutionCrossroads/si_us_revolutionary_era_collections (7.7 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_OCR_results/viewer/block_level (19.5 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_NER_Entity_Occurrences (939 MB)

In [ ]:
# Set up dataset stream
ocr_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                      name="page_level",
                      split="train",
                      streaming=True)
ocr_ds.features

In [ ]:
# For filtering data stream
def is_pension(example):
  return example["collection"] == "rev_war_pension"
def is_news(example):
  return example["collection"] == "chronicling_america"

In [ ]:
# Filter stream and get 100 rows of text
news_ds = ocr_ds.filter(is_news)
news_100_ds = news_ds.take(100)
news_100_text = list(news_100_ds["new_ocr_text"])

In [ ]:
# Count terms
tf_vectorizer = CountVectorizer(max_df=0.95,
                                min_df=2,
                                ngram_range=(1, 2),
                                max_features=1000,
                                stop_words="english")

tf = tf_vectorizer.fit_transform(news_100_text)
tf_term_names = tf_vectorizer.get_feature_names_out()

In [ ]:
# Put term counts in DataFrame
tf_df = pd.DataFrame(tf.todense(), columns=tf_term_names)

# Sum counts across documents, sort by counts, get top 10 words
tf_df.sum().sort_values(ascending=False).head(10)

In [ ]:
# Calculate term frequencies
tfidf_vectorizer = TfidfVectorizer(max_df=0.95,
                                   min_df=2,
                                   ngram_range=(1, 2),
                                   max_features=1000,
                                   stop_words="english")

tfidf = tfidf_vectorizer.fit_transform(news_100_text)
tfidf_term_names = tfidf_vectorizer.get_feature_names_out()

In [ ]:
# Put term scores in DataFrame
tfidf_df = pd.DataFrame(tfidf.todense(), columns=tfidf_term_names)

# Sum counts across documents, sort by counts, get top 10 terms
tfidf_df.sum().sort_values(ascending=False).head(10)

In [ ]:
# Use term counts for LDA
n_topics = 4

lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=10,
    learning_method="online",
    learning_offset=1.0,
    random_state=1010,
)
lda = lda.fit(tf)

In [ ]:
# Print topic words
words_per_topic = 8
word_offset = 2

for topic_idx, topic in enumerate(lda.components_):
  top_features_ind = topic.argsort()[-words_per_topic-word_offset:-word_offset]
  top_features = tf_term_names[top_features_ind]
  print(f"t{topic_idx}: {top_features}")

## BERTopic

In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, PartOfSpeech, ZeroShotClassification

In [ ]:
representation_model = KeyBERTInspired()
topic_model = BERTopic(representation_model=representation_model, language="english")
topics, probs = topic_model.fit_transform(news_100_text)
topic_model.get_topic_info()

In [ ]:
pos_patterns = [
  [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
  [{'POS': 'NOUN'}],
  [{'POS': 'ADJ'}]
]

representation_model = PartOfSpeech("en_core_web_sm", pos_patterns=pos_patterns)
topic_model = BERTopic(representation_model=representation_model, language="english")
topics, probs = topic_model.fit_transform(news_100_text)
topic_model.get_topic_info()

In [ ]:
# Chaining multiple representations together
candidate_topics = ["commerce", "politics", "tariffs", "war"]
main_representation = KeyBERTInspired()
zsc_model = ZeroShotClassification(candidate_topics, model="facebook/bart-large-mnli")

representation_model = {
  "Main": main_representation,
  "ZeroShot":  zsc_model,
}

topic_model = BERTopic(representation_model=representation_model, language="english")
topics, probs = topic_model.fit_transform(news_100_text)
topic_model.get_topic_info()